## True vs predicted CWEB classes (subvolume)

This notebook:

- Loads an **SBI cache** (pickle) that contains `graph`, `classification_labels`, and `masks`.
- Loads the **best checkpoint** produced by `workflows/jraph/jraph_pipeline.py`.
- Runs inference to produce **predicted cosmic-web class** for every node in the subvolume.
- Plots two **interactive 3D** views (true vs predicted) using Plotly.

### Inputs you must set

- `CACHE_PATH`: classification cache `.pkl` used for training
- `RUN_DIR`: output directory that contains `checkpoints/best_checkpoint.json`
- `POINTS_XYZ_PATH`: `*_points_xyz.npy` aligned to the cache node order (same subvolume graph)


In [ ]:
from __future__ import annotations

import json
import os
import pickle
import sys
from pathlib import Path

import numpy as np

# --- JAX backend control (must run BEFORE importing jax later) ---
# Full-graph inference frequently OOMs on GPUs/login environments.
# Force CPU unless you intentionally want GPU.
os.environ.setdefault("JAX_PLATFORMS", "cpu")
os.environ.setdefault("JAX_PLATFORM_NAME", "cpu")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

# Ensure imports work when running from `.../workflows/abacus_tweb`.
# We need the directory that contains `workflows/` on sys.path.
CWD = Path.cwd().resolve()
root = CWD
ILLUSTRIS_ROOT = None
for _ in range(8):
    if (root / "workflows").is_dir():
        ILLUSTRIS_ROOT = root
        break
    root = root.parent
if ILLUSTRIS_ROOT is None:
    raise RuntimeError(f"Could not find repo root containing `workflows/` from CWD={CWD}")
if str(ILLUSTRIS_ROOT) not in sys.path:
    sys.path.insert(0, str(ILLUSTRIS_ROOT))

print("CWD:", CWD)
print("ILLUSTRIS_ROOT:", ILLUSTRIS_ROOT)
print("JAX_PLATFORMS:", os.environ.get("JAX_PLATFORMS"))

# --- User inputs (prefilled from your rs8 classification run) ---
CACHE_PATH = Path(
    "/pscratch/sd/d/dkololgi/abacus/sbi_caches/abacus_delaunay_cube_x850_1150_y-150_150_z208_508_mpc_h_sbi_cache_rs8_ngrid2048_thr0p2_15d.pkl"
)
RUN_DIR = Path(
    "/pscratch/sd/d/dkololgi/abacus/jraph_runs/cube_classification_rs8_ngrid2048_thr0p2_from15dcache"
)

# IMPORTANT: this must be the points_xyz for the SAME cube graph as the cache.
POINTS_XYZ_PATH = Path(
    "/pscratch/sd/d/dkololgi/abacus/graph_constructions/abacus_delaunay_cube_x850_1150_y-150_150_z208_508_mpc_h_points_xyz.npy"
)

# Optional: downsample for plotting responsiveness
PLOT_MAX_N = 200_000
RNG_SEED = 0


In [ ]:
import jax
import jax.numpy as jnp
import haiku as hk

print("JAX devices:", jax.devices())

# Reuse the exact network definition from the training script.
# (This import works when `.../TNG/Illustris` is on sys.path; see cell above.)
from workflows.jraph.jraph_pipeline import make_graph_network  # type: ignore


def load_cache(path: Path) -> dict:
    with path.open("rb") as f:
        d = pickle.load(f)
    if "graph" not in d:
        raise KeyError(f"Cache missing 'graph': {path}")
    if "classification_labels" not in d:
        raise KeyError(
            "Cache missing 'classification_labels'. This notebook is for classification runs. "
            "(If you trained regression, use eigenvalues_raw instead.)"
        )
    return d


def load_best_checkpoint(run_dir: Path) -> dict:
    pointer = run_dir / "checkpoints" / "best_checkpoint.json"
    if not pointer.exists():
        raise FileNotFoundError(f"Missing: {pointer}")
    info = json.loads(pointer.read_text())
    ckpt_path = Path(info["path"]).expanduser()
    with ckpt_path.open("rb") as f:
        ckpt = pickle.load(f)
    if "params" not in ckpt:
        raise KeyError(f"Checkpoint missing 'params': {ckpt_path}")
    return {"info": info, "path": ckpt_path, "ckpt": ckpt}


cache = load_cache(CACHE_PATH)
graph = cache["graph"]
y_true = np.asarray(cache["classification_labels"], dtype=np.int32)

ckpt_bundle = load_best_checkpoint(RUN_DIR)
ckpt = ckpt_bundle["ckpt"]
params = ckpt["params"]

print("Loaded cache:", CACHE_PATH)
print("  nodes:", int(graph.n_node[0]), "edges:", int(graph.n_edge[0]))
print("  y_true shape:", y_true.shape, "classes:", np.unique(y_true))
print("Loaded checkpoint:", ckpt_bundle["path"])
print("  epoch:", ckpt.get("epoch"), "prediction_mode:", ckpt.get("prediction_mode"))


In [ ]:
# Reconstruct the same Haiku-transformed model and run inference.
# These MUST match the training run hyperparameters.
# From your rs16 classification command:
#   --latent_size 96 --num_heads 8 --num_passes 8 --dropout 0.15
NUM_PASSES = 8
LATENT_SIZE = 96
NUM_HEADS = 8
DROPOUT = 0.15
OUTPUT_DIM = 4

net_fn = make_graph_network(
    num_passes=NUM_PASSES,
    latent_size=LATENT_SIZE,
    num_heads=NUM_HEADS,
    dropout_rate=DROPOUT,
    output_dim=OUTPUT_DIM,
)
net = hk.transform(net_fn)

rng = jax.random.PRNGKey(RNG_SEED)
logits = net.apply(params, rng, graph, is_training=False)

# jraph_pipeline uses `.nodes` from the returned GraphsTuple; keep compatibility here.
try:
    node_logits = np.asarray(logits.nodes)
except Exception:
    # If make_graph_network returns raw node logits, accept that.
    node_logits = np.asarray(logits)

y_pred = node_logits.argmax(axis=-1).astype(np.int32)

print("logits:", node_logits.shape)
print("y_pred:", y_pred.shape, "classes:", np.unique(y_pred))
print("Accuracy (all nodes):", float((y_pred == y_true).mean()))


In [ ]:
# Load 3D coordinates aligned to node order.
xyz = np.load(POINTS_XYZ_PATH)
if xyz.ndim != 2 or xyz.shape[1] != 3:
    raise ValueError(f"Expected (N,3) xyz, got {xyz.shape} from {POINTS_XYZ_PATH}")

n = int(graph.n_node[0])
if xyz.shape[0] != n:
    raise ValueError(
        f"points_xyz length mismatch: xyz has {xyz.shape[0]:,} rows but cache graph has {n:,} nodes.\n"
        "You must use the points_xyz for the SAME subvolume graph that produced this cache."
    )

# Optional downsample for Plotly responsiveness
idx = np.arange(n)
if PLOT_MAX_N is not None and n > int(PLOT_MAX_N):
    rng_np = np.random.default_rng(RNG_SEED)
    idx = rng_np.choice(idx, size=int(PLOT_MAX_N), replace=False)

xyz_p = xyz[idx]
true_p = y_true[idx]
pred_p = y_pred[idx]

print("Plotting N=", xyz_p.shape[0])


In [ ]:
import plotly.express as px

CLASS_NAMES = np.array(["Void", "Wall", "Filament", "Cluster"], dtype=object)

# Stable discrete mapping
COLOR_MAP = {
    0: "#1f77b4",  # blue
    1: "#ff7f0e",  # orange
    2: "#2ca02c",  # green
    3: "#d62728",  # red
}

def make_3d_fig(xyz3: np.ndarray, cls: np.ndarray, *, title: str):
    cls = np.asarray(cls, dtype=np.int32)
    df = {
        "x": xyz3[:, 0],
        "y": xyz3[:, 1],
        "z": xyz3[:, 2],
        "class": [CLASS_NAMES[c] if 0 <= c < len(CLASS_NAMES) else f"{c}" for c in cls],
        "class_id": cls,
    }
    fig = px.scatter_3d(
        df,
        x="x",
        y="y",
        z="z",
        color="class",
        color_discrete_map={CLASS_NAMES[k]: v for k, v in COLOR_MAP.items()},
        opacity=0.75,
        title=title,
    )
    fig.update_traces(marker=dict(size=2))
    fig.update_layout(
        scene=dict(
            xaxis_title="x [Mpc]",
            yaxis_title="y [Mpc]",
            zaxis_title="z [Mpc]",
            aspectmode="data",
        ),
        legend_title_text="CWEB",
    )
    return fig

fig_true = make_3d_fig(xyz_p, true_p, title="True CWEB classes (subvolume)")
fig_pred = make_3d_fig(xyz_p, pred_p, title="Predicted CWEB classes (subvolume)")

fig_true.show()
fig_pred.show()
